In [ ]:
import pandas as pd
from pathlib import Path
import ast

# Display all rows
pd.set_option("display.max_rows", None)

# Display all columns
pd.set_option("display.max_columns", None)

# Display full column width (prclickstream truncation of long strings/text)
pd.set_option("display.max_colwidth", None)

# Ensure the output width accommodates all columns in a console
pd.set_option("display.width", None)

In [2]:
data_path = Path("../data/raw")

In [3]:
clickstream = pd.read_csv(data_path / "click_stream.csv")

### Event‑Type Breakdown:
HOMEPAGE > None

SCROLL	> None

SEARCH	> search_keywords

ITEM_DETAIL	> product_id, quantity, item_price

ADD_TO_CART	> product_id, quantity, item_price

BOOKING	> payment_status (and sometimes promo_code, promo_amount)	

Other events	> None	Events like BANNER_CLICK, CATEGORY_VIEW, etc., have no metadata.

In [4]:
# --- PHASE 1: PRE-TRANSFORMATION OPTIMIZATION ---
# Convert time strings to actual datetime objects
clickstream["event_time"] = pd.to_datetime(clickstream["event_time"])

# Convert low-cardinality strings to Categories to save RAM
cat_cols = ["event_name", "traffic_source"]
for col in cat_cols:
    clickstream[col] = clickstream[col].astype("category")

# --- PHASE 2: METADATA FLATTENING ---
# 1. Handle NaNs and ensure string type for evaluation
clickstream["event_metadata"] = clickstream["event_metadata"].fillna("{}").astype(str)

# 2. Convert strings to actual Python dictionaries (Fast list comprehension)
clickstream["event_metadata"] = [
    ast.literal_eval(x) for x in clickstream["event_metadata"]
]

# 3. Flatten the dictionaries into a temporary DataFrame
meta_df = pd.DataFrame(clickstream["event_metadata"].tolist(), index=clickstream.index)

# 4. Concatenate back to clickstream and drop the old metadata column
clickstream = pd.concat([clickstream.drop("event_metadata", axis=1), meta_df], axis=1)

# --- PHASE 3: POST-TRANSFORMATION TYPE ENFORCEMENT ---
# Use 'Int64' (Capital I) to allow Integers AND NaNs (Sparse columns)
if "product_id" in clickstream.columns:
    clickstream["product_id"] = pd.to_numeric(
        clickstream["product_id"], errors="coerce"
    ).astype("Int64")

if "quantity" in clickstream.columns:
    clickstream["quantity"] = pd.to_numeric(
        clickstream["quantity"], errors="coerce"
    ).astype("Int64")

# Categorize the payment status
if "payment_status" in clickstream.columns:
    clickstream["payment_status"] = clickstream["payment_status"].astype("category")

In [5]:
session_times = clickstream.groupby("session_id")["event_time"].agg(["min", "max"])
session_times["duration_seconds"] = (
    session_times["max"] - session_times["min"]
).dt.total_seconds()

In [6]:
clickstream.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12833602 entries, 0 to 12833601
Data columns (total 12 columns):
 #   Column           Dtype              
---  ------           -----              
 0   session_id       object             
 1   event_name       category           
 2   event_time       datetime64[ns, UTC]
 3   event_id         object             
 4   traffic_source   category           
 5   product_id       Int64              
 6   quantity         Int64              
 7   item_price       float64            
 8   payment_status   category           
 9   search_keywords  object             
 10  promo_code       object             
 11  promo_amount     float64            
dtypes: Int64(2), category(3), datetime64[ns, UTC](1), float64(2), object(4)
memory usage: 942.4+ MB


In [7]:
clickstream[~clickstream["search_keywords"].isna()].head()

,session_id,event_name,event_time,event_id,traffic_source,product_id,quantity,item_price,payment_status,search_keywords,promo_code,promo_amount
5,7d440441-e67a-4d36-b324-80ffd636d166,SEARCH,2019-09-01 12:07:00.322763+00:00,c952142b-4fe9-4694-ad7f-21a5d1bed9ca,MOBILE,<NA>,<NA>,NaN,NaN,Dress Kondangan,NaN,NaN
22,f7b5cd60-4f87-4b3c-8bde-c7da8930eec2,SEARCH,2019-09-09 05:06:26.425431+00:00,65ad97d0-9207-447e-b09d-fcc00a569824,MOBILE,<NA>,<NA>,NaN,NaN,Tas Wanita,NaN,NaN
35,f7b5cd60-4f87-4b3c-8bde-c7da8930eec2,SEARCH,2019-09-09 08:50:27.425431+00:00,a2e93548-18b5-4a79-b74f-42bbc0cf0fbd,MOBILE,<NA>,<NA>,NaN,NaN,Celana Panjang,NaN,NaN
57,98e66f53-f24f-4238-8c9c-34a0482e22d5,SEARCH,2019-10-02 17:17:08.425431+00:00,bf6963cd-b681-4763-b796-ce18d3b9004f,MOBILE,<NA>,<NA>,NaN,NaN,Kaos,NaN,NaN
59,98e66f53-f24f-4238-8c9c-34a0482e22d5,SEARCH,2019-10-02 21:07:33.425431+00:00,cf5d38ae-7f44-4166-9cf9-c43fb57b8f3a,MOBILE,<NA>,<NA>,NaN,NaN,Sepatu,NaN,NaN


In [8]:
clickstream["item_price"] = clickstream["item_price"].astype("float32")
clickstream["promo_amount"] = clickstream["promo_amount"].astype("float32")
clickstream["product_id"] = clickstream["product_id"].astype("Int32")
clickstream["quantity"] = clickstream["quantity"].astype("Int32")

clickstream["event_hour"] = clickstream["event_time"].dt.hour
clickstream["event_dayofweek"] = clickstream["event_time"].dt.dayofweek
clickstream["event_date"] = clickstream["event_time"].dt.normalize()

# 2. Convert promo_code to category (after filling NaNs with a placeholder)
clickstream["promo_code"] = (
    clickstream["promo_code"].fillna("NO_PROMO").astype("category")
)
# Standardize search keywords (lowercase and strip spaces)
if "search_keywords" in clickstream.columns:
    clickstream["search_keywords"] = (
        clickstream["search_keywords"].str.lower().str.strip()
    )

In [9]:
# Check nulls in critical columns
critical_cols = ["session_id", "event_name", "event_time", "product_id"]
null_counts = clickstream[critical_cols].isnull().sum()
print("Nulls in critical columns:\n", null_counts)

Nulls in critical columns:
 session_id           0
event_name           0
event_time           0
product_id    10896445
dtype: int64


In [10]:
clickstream["event_time"].max()

Timestamp('2022-08-01 00:49:59.408424+0000', tz='UTC')

In [11]:
clickstream.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12833602 entries, 0 to 12833601
Data columns (total 15 columns):
 #   Column           Dtype              
---  ------           -----              
 0   session_id       object             
 1   event_name       category           
 2   event_time       datetime64[ns, UTC]
 3   event_id         object             
 4   traffic_source   category           
 5   product_id       Int32              
 6   quantity         Int32              
 7   item_price       float32            
 8   payment_status   category           
 9   search_keywords  object             
 10  promo_code       category           
 11  promo_amount     float32            
 12  event_hour       int32              
 13  event_dayofweek  int32              
 14  event_date       datetime64[ns, UTC]
dtypes: Int32(2), category(4), datetime64[ns, UTC](2), float32(2), int32(2), object(3)
memory usage: 856.7+ MB


In [14]:
# Missing values
print("Missing values per column:\n", clickstream.isnull().sum())

Missing values per column:
 session_id                0
event_name                0
event_time                0
event_id                  0
traffic_source            0
product_id         10896445
quantity           10896445
item_price         10896445
payment_status     11981020
search_keywords    11660336
promo_code                0
promo_amount       12507067
event_hour                0
event_dayofweek           0
event_date                0
dtype: int64


In [12]:
# Duplicate event_id? (should be unique)
duplicate_events = clickstream["event_id"].duplicated().sum()
print(f"Duplicate event_id: {duplicate_events}")

Duplicate event_id: 0


In [18]:
clickstream["event_name"].value_counts()

event_name
CLICK          2498038
HOMEPAGE       2487126
ADD_TO_CART    1937157
SCROLL         1663571
ITEM_DETAIL    1290396
SEARCH         1173266
BOOKING         852582
PROMO_PAGE      604931
ADD_PROMO       326535
Name: count, dtype: int64

In [13]:
clickstream.to_parquet(
    "../data/interim/clickstream_clean.parquet", index=False, compression="snappy"
)